# Simple MLP for text classification from scratch

__Data__

In [ ]:
import numpy as np
import copy
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Set maximum number of features
max_features = 1000
# Choose two categories for classification (for all available categories, see
# https://scikit-learn.org/stable/datasets/real_world.html#usage)
categories = ["sci.space", "talk.politics.misc"]

train_raw = fetch_20newsgroups(
    subset="train", categories=categories, remove=("headers", "footers", "quotes")
)
test_raw = fetch_20newsgroups(
    subset="test", categories=categories, remove=("headers", "footers", "quotes")
)

vectorizer = TfidfVectorizer(max_features=max_features)
X_train = vectorizer.fit_transform(train_raw.data).toarray()
X_test = vectorizer.transform(test_raw.data).toarray()

# Labels as column vectors: 0 = sci.space, 1 = talk.politics.misc
y_train = train_raw.target.reshape(-1, 1).astype(float)
y_test = test_raw.target.reshape(-1, 1).astype(float)

# Standardise features using only training statistics (avoids data leakage)
mean_X = X_train.mean(axis=0)
sd_X = X_train.std(axis=0)
sd_X[sd_X == 0] = 1  # avoid division by zero for constant features if any

X_train = (X_train - mean_X) / sd_X
X_test = (X_test - mean_X) / sd_X

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Class balance (train): {y_train.mean():.2f}  (test): {y_test.mean():.2f}")

__Neural network__

One hidden layer with sigmoid activation, and a sigmoid output for binary classification:

$$z^{(1)} = X W^{(1)} + b^{(1)}, \quad a^{(1)} = \sigma(z^{(1)})$$
$$z^{(2)} = a^{(1)} W^{(2)} + b^{(2)}, \quad \hat{y} = \sigma(z^{(2)})$$

In [ ]:
input_dim = X_train.shape[1]
hidden_layer_size = 64

# Initialise weights and biases
np.random.seed(42)
weights_init = {
    "W1": np.random.normal(0, 0.01, size=(input_dim, hidden_layer_size)),
    "b1": np.zeros(hidden_layer_size),
    "W2": np.random.normal(0, 0.01, size=(hidden_layer_size, 1)),
    "b2": np.zeros(1),
}

__Loss function and gradients__

$$J(\theta) = -\sum \left[ y \log \hat{y} + (1 - y) \log (1 - \hat{y}) \right]$$

With a sigmoid output $\hat{y} = \sigma(z^{(2)})$, the output-layer error simplifies elegantly:

$$\delta^{(2)} = \hat{y} - y$$

The full gradients are:

$$\frac{\partial J}{\partial W^{(2)}} = a^{(1)T} \delta^{(2)}$$

$$\frac{\partial J}{\partial b^{(2)}} = \mathbf{1}^T \delta^{(2)}$$

$$\frac{\partial J}{\partial W^{(1)}} = X^T \left( \delta^{(2)} W^{(2)T} \odot \sigma'(z^{(1)}) \right)$$

$$\frac{\partial J}{\partial b^{(1)}} = \mathbf{1}^T \left( \delta^{(2)} W^{(2)T} \odot \sigma'(z^{(1)}) \right)$$

In [ ]:
def sigmoid(x):
    x = np.clip(x, -500, 500)  # avoid overflow in exp
    return 1 / (1 + np.exp(-x))


def sigmoid_prime(x):
    s = sigmoid(x)
    return s * (1 - s)


def forward_pass(X, weights):
    z1 = X @ weights["W1"] + weights["b1"]
    a1 = sigmoid(z1)
    z2 = a1 @ weights["W2"] + weights["b2"]
    y_hat = sigmoid(z2)
    return y_hat


def predict(X, weights, threshold=0.5):
    return (forward_pass(X, weights) >= threshold).astype(float)


def J(X, y, weights):
    y_hat = np.clip(forward_pass(X, weights), 1e-12, 1 - 1e-12)
    return -np.sum(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat)) / len(y)


def J_prime(X, y, weights):
    z1 = X @ weights["W1"] + weights["b1"]
    a1 = sigmoid(z1)
    z2 = a1 @ weights["W2"] + weights["b2"]
    y_hat = sigmoid(z2)

    delta2 = y_hat - y  # (n, 1)
    delta1 = (delta2 @ weights["W2"].T) * sigmoid_prime(z1)  # (n, hidden)

    dJdW2 = a1.T @ delta2  # (hidden, 1)
    dJdb2 = np.ones(len(y)) @ delta2  # (1,)
    dJdW1 = X.T @ delta1  # (input, hidden)
    dJdb1 = np.ones(len(y)) @ delta1  # (hidden,)

    return {"dJdW1": dJdW1, "dJdb1": dJdb1, "dJdW2": dJdW2, "dJdb2": dJdb2}

__Training__

In [ ]:
def train(X, y, weights, n_epochs, batch_size, learning_rate):
    """
    Train the network with mini-batch gradient descent.
    Returns trained weights; original weights are not mutated.
    """
    weights = copy.deepcopy(weights)
    n = len(X)

    acc = (predict(X, weights) == y).mean()
    print(f"Start | Loss: {J(X, y, weights):.4f} | Train accuracy: {acc:.4f}")

    for epoch in range(1, n_epochs + 1):
        indices = np.random.permutation(n)

        for start in range(0, n, batch_size):
            batch_idx = indices[start : start + batch_size]
            X_batch = X[batch_idx]
            y_batch = y[batch_idx]

            grads = J_prime(X_batch, y_batch, weights)

            weights["W1"] -= learning_rate * grads["dJdW1"] / batch_size
            weights["b1"] -= learning_rate * grads["dJdb1"] / batch_size
            weights["W2"] -= learning_rate * grads["dJdW2"] / batch_size
            weights["b2"] -= learning_rate * grads["dJdb2"] / batch_size

        acc = (predict(X, weights) == y).mean()
        print(
            f"Epoch {epoch} | Loss: {J(X, y, weights):.4f} | Train accuracy: {acc:.4f}"
        )

    return weights

In [ ]:
np.random.seed(123)

weights_trained = train(
    X=X_train,
    y=y_train,
    weights=weights_init,
    n_epochs=10,
    batch_size=64,
    learning_rate=0.15,
)

__Evaluation__

In [ ]:
# Benchmark: logistic regression
lr = LogisticRegression(C=np.inf, max_iter=1000)  # C=np.inf disables regularisation
lr.fit(X_train, y_train.ravel())
y_hat_lr = lr.predict(X_test).reshape(-1, 1)

# Neural network
y_hat_before_training = predict(X_test, weights_init)
y_hat_after_training = predict(X_test, weights_trained)

print(
    f"Logistic regression test accuracy:              {(y_hat_lr == y_test).mean():.3f}"
)
print(
    f"Neural network test accuracy (before training): {((y_hat_before_training == y_test).mean()):.3f}"
)
print(
    f"Neural network test accuracy (after training):  {((y_hat_after_training == y_test).mean()):.3f}"
)

## References

- https://github.com/stephencwelch/Neural-Networks-Demystified (in particular Part 4)
- https://www.3blue1brown.com/lessons/backpropagation-calculus